# Lesson 6: Dataset, DataLoader and categorical embeddings

Goes with the Lesson 6 page of the Forkcast course. Real taxi trips as a stand-in for deliveries: cleaning, a `Dataset`, a `DataLoader`, mini-batches, and `nn.Embedding` for zone ids.

Runtime: CPU is enough for the bundled sample; a GPU makes the real month faster at the end.

## 1. Clone your repository and install the package

In [ ]:
GITHUB_USER = "your-github-username"   # edit this line
REPO = "forkcast-course"

!git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!pip install -e .

## 2. Load, clean, split

`clean_trips` and `load_trips` are provided. Read them: they print what they remove.

In [ ]:
%%writefile forkcast/trips.py
"""Lesson 6. Real deliveries: taxi trips as a stand-in for Forkcast ETAs.

``clean_trips`` and ``load_trips`` are provided and apply the same rules to the
bundled sample and to a real month. ``TripsDataset`` and ``make_loaders`` are
the exercises.
"""

from __future__ import annotations

from pathlib import Path

import pandas as pd
import torch
from torch import Tensor
from torch.utils.data import DataLoader, Dataset

CONTINUOUS = ["trip_distance", "pickup_hour", "weekday", "passenger_count"]
CATEGORICAL = ["pu_zone", "do_zone"]
ZONE_VOCAB = 266          # TLC zone ids run 1..265; index 0 is reserved for "unknown"
TARGET = "minutes"
SAMPLE = Path(__file__).resolve().parent.parent / "data" / "taxi_sample.csv"


def clean_trips(raw: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """Turn TLC columns into model columns and drop impossible rows.

    Rules, stated once so they can be argued with: a trip must move (distance 0 to 50 miles),
    take between 1 and 120 minutes, and have a known or explicitly unknown zone.
    """
    frame = raw.copy()
    pickup = pd.to_datetime(frame["tpep_pickup_datetime"])
    dropoff = pd.to_datetime(frame["tpep_dropoff_datetime"])
    frame[TARGET] = (dropoff - pickup).dt.total_seconds() / 60
    frame["pickup_hour"] = pickup.dt.hour.astype("float32")
    frame["weekday"] = pickup.dt.dayofweek.astype("float32")
    frame["passenger_count"] = frame["passenger_count"].fillna(1).clip(1, 6).astype("float32")
    frame["trip_distance"] = frame["trip_distance"].astype("float32")
    for src, dst in (("PULocationID", "pu_zone"), ("DOLocationID", "do_zone")):
        zone = frame[src].fillna(0).astype(int)
        frame[dst] = zone.where((zone >= 1) & (zone <= 265), 0)          # unknown -> 0
    before = len(frame)
    keep = (frame["trip_distance"] > 0) & (frame["trip_distance"] <= 50) & (frame[TARGET] >= 1) & (frame[TARGET] <= 120)
    frame = frame.loc[keep, CONTINUOUS + CATEGORICAL + [TARGET]].reset_index(drop=True)
    if verbose:
        print(f"clean_trips: kept {len(frame):,} of {before:,} rows ({before - len(frame):,} removed)")
    return frame


def load_trips(path: str | Path = SAMPLE, verbose: bool = True) -> pd.DataFrame:
    """Read a CSV or parquet of TLC trips and clean it."""
    path = Path(path)
    raw = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
    return clean_trips(raw, verbose=verbose)


def split_frame(frame: pd.DataFrame, fractions=(0.7, 0.15, 0.15), seed: int = 0):
    """Shuffle rows and cut into three frames (Lesson 5's split, for DataFrames)."""
    perm = torch.randperm(len(frame), generator=torch.Generator().manual_seed(seed)).numpy()
    n_train, n_val = round(len(frame) * fractions[0]), round(len(frame) * fractions[1])
    parts = perm[:n_train], perm[n_train:n_train + n_val], perm[n_train + n_val:]
    return tuple(frame.iloc[idx].reset_index(drop=True) for idx in parts)


class TripsDataset(Dataset):
    """One trip per item: (continuous features (4,), zone ids (2,) as int64, minutes ()).

    ``normalizer`` is a fitted Normalizer for the continuous columns (fit it on the
    training frame only). Store everything as tensors in ``__init__`` so ``__getitem__``
    is a plain index.
    """

    def __init__(self, frame: pd.DataFrame, normalizer=None):
        raise NotImplementedError("Lesson 6, step 3: cont = torch.tensor(frame[CONTINUOUS].values, dtype=torch.float32), normalized; cat = ... .long(); target = ...")

    def __len__(self) -> int:
        raise NotImplementedError("Lesson 6, step 3: how many trips?")

    def __getitem__(self, i: int):
        raise NotImplementedError("Lesson 6, step 3: return self.cont[i], self.cat[i], self.target[i]")


def make_loaders(train_frame: pd.DataFrame, val_frame: pd.DataFrame, normalizer, batch_size: int = 256) -> tuple[DataLoader, DataLoader]:
    """Training loader: shuffled, drop_last=True. Validation loader: in order, every row."""
    raise NotImplementedError("Lesson 6, step 4: two TripsDatasets, two DataLoaders")

In [ ]:
import importlib, torch
from torch import nn
import matplotlib.pyplot as plt
from forkcast import trips
importlib.reload(trips)
from forkcast.preprocess import Normalizer

frame = trips.load_trips()
print(frame.head(3))
print("minutes: mean", round(frame["minutes"].mean(), 1), "std", round(frame["minutes"].std(), 1))
train_frame, val_frame, test_frame = trips.split_frame(frame, seed=0)
print(len(train_frame), len(val_frame), len(test_frame))
norm = Normalizer().fit(torch.tensor(train_frame[trips.CONTINUOUS].values, dtype=torch.float32))
print(norm.mean)

## 3. `TripsDataset` and `make_loaders`

Fill in the four `raise NotImplementedError` lines in the `trips.py` cell above and re-run it, then check:

In [ ]:
importlib.reload(trips)
ds = trips.TripsDataset(train_frame, norm)
print(len(ds)); print(ds[0])
train_loader, val_loader = trips.make_loaders(train_frame, val_frame, norm, batch_size=256)
print(len(train_loader), len(val_loader))
cont, cat, minutes = next(iter(train_loader))
print(cont.shape, cat.shape, minutes.shape, cat.dtype)

## 4. `TabularNet`: `forkcast/model.py`

In [ ]:
%%writefile forkcast/model.py
"""Lesson 3. The delivery-time model as an nn.Module (reference solution)."""

from __future__ import annotations

import torch
from torch import Tensor, nn


class LinearRegressor(nn.Module):
    """minutes = X @ w + b, with the knobs owned by an nn.Linear layer."""

    def __init__(self, n_features: int):
        super().__init__()                        # registers parameters, enables .to(), state_dict, ...
        self.linear = nn.Linear(n_features, 1)    # weight (1, d), bias (1,)

    def forward(self, X: Tensor) -> Tensor:
        """(N, d) -> (N,)"""
        return self.linear(X).squeeze(-1)         # (N, 1) -> (N,), so it lines up with y

# ---- Lesson 4: the first neural network -------------------------------------

def count_parameters(model: nn.Module) -> int:
    """Number of trainable numbers in ``model``."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class MLP(nn.Module):
    """A multilayer perceptron: Linear -> activation [-> Dropout] -> ... -> Linear(…, 1).

    ``dropout`` (Lesson 5) is the fraction of hidden units zeroed on each training step; 0 adds no Dropout layers.
    """

    def __init__(self, n_features: int, hidden: tuple[int, ...] = (32, 32), activation=nn.ReLU, dropout: float = 0.0):
        super().__init__()
        layers: list[nn.Module] = []
        width = n_features
        for h in hidden:
            layers.append(nn.Linear(width, h))        # (N, width) -> (N, h)
            layers.append(activation())               # bends; without it two Linears are one Linear
            if dropout > 0:
                layers.append(nn.Dropout(dropout))    # training only: randomly zero a fraction of the units
            width = h
        layers.append(nn.Linear(width, 1))            # the output layer has no activation
        self.net = nn.Sequential(*layers)

    def forward(self, X: Tensor) -> Tensor:
        """(N, d) -> (N,)"""
        return self.net(X).squeeze(-1)


# ---- Lesson 6: continuous features plus embedded zones ------------------------

class TabularNet(nn.Module):
    """Continuous features and two zone ids -> minutes.

    Each zone id (0..zone_vocab-1) is looked up in an nn.Embedding table of shape
    (zone_vocab, emb_dim). The two embedded zones are concatenated with the
    continuous features and fed to an MLP: input width n_continuous + 2 * emb_dim.
    """

    def __init__(self, n_continuous: int, zone_vocab: int = 266, emb_dim: int = 8, hidden: tuple[int, ...] = (64, 64)):
        super().__init__()
        raise NotImplementedError("Lesson 6, step 5: self.pu = nn.Embedding(zone_vocab, emb_dim); self.do = ...; self.mlp = MLP(n_continuous + 2 * emb_dim, hidden)")

    def forward(self, x_cont: Tensor, x_cat: Tensor) -> Tensor:
        """(N, n_continuous) float, (N, 2) int64 -> (N,)"""
        raise NotImplementedError("Lesson 6, step 5: torch.cat([x_cont, self.pu(x_cat[:, 0]), self.do(x_cat[:, 1])], dim=1) -> self.mlp")

In [ ]:
from forkcast import model as model_module
importlib.reload(model_module)
from forkcast.model import TabularNet, MLP, count_parameters
model = TabularNet(4)
print(model(cont, cat).shape)             # torch.Size([256])
print(count_parameters(model))            # 9825
print(model.pu.weight.shape)              # torch.Size([266, 8])

## 5. The toolkit learns to take loaders: `forkcast/train.py`

In [ ]:
%%writefile forkcast/train.py
"""Lesson 3. The training toolkit: one epoch, evaluation, fit, save, load (reference solution)."""

from __future__ import annotations

from pathlib import Path

import torch
from torch import Tensor, nn


def train_one_epoch(model: nn.Module, X: Tensor, y: Tensor, optimizer: torch.optim.Optimizer, loss_fn: nn.Module) -> float:
    """One pass over the data. Returns the loss as a Python float."""
    model.train()
    optimizer.zero_grad()                 # 1. clean slate (Lesson 2's zero_() for every parameter)
    loss = loss_fn(model(X), y)           #    forward
    loss.backward()                       # 2. slopes into every parameter's .grad
    optimizer.step()                      # 3. the update rule, applied to every parameter
    return loss.item()


@torch.no_grad()
def evaluate(model: nn.Module, X: Tensor, y: Tensor, loss_fn: nn.Module) -> float:
    """Loss on data the model is not learning from."""
    model.eval()
    return loss_fn(model(X), y).item()


def fit(model: nn.Module, X: Tensor, y: Tensor, epochs: int = 300, lr: float = 0.1,
        optimizer_cls=torch.optim.SGD, loss_fn: nn.Module | None = None) -> list[float]:
    """Build the optimizer, run ``epochs`` epochs, return one loss per epoch."""
    loss_fn = loss_fn or nn.MSELoss()
    optimizer = optimizer_cls(model.parameters(), lr=lr)
    return [train_one_epoch(model, X, y, optimizer, loss_fn) for _ in range(epochs)]


def save(model: nn.Module, path: str | Path) -> None:
    torch.save(model.state_dict(), path)


def load(model: nn.Module, path: str | Path) -> nn.Module:
    model.load_state_dict(torch.load(path, weights_only=True))
    return model


# ---- Lesson 5: validation, early stopping, regularization ------------------

def fit_with_validation(model: nn.Module, X_train: Tensor, y_train: Tensor, X_val: Tensor, y_val: Tensor,
                        epochs: int = 300, lr: float = 0.01, optimizer_cls=torch.optim.Adam,
                        weight_decay: float = 0.0, patience: int | None = None, loss_fn: nn.Module | None = None) -> dict:
    """Train while watching the validation loss; keep the best weights; optionally stop early."""
    import copy

    loss_fn = loss_fn or nn.MSELoss()
    optimizer = optimizer_cls(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {"train": [], "val": [], "best_epoch": 0, "best_val": float("inf")}
    best_state = copy.deepcopy(model.state_dict())
    since_best = 0
    for epoch in range(epochs):
        history["train"].append(train_one_epoch(model, X_train, y_train, optimizer, loss_fn))
        val_loss = evaluate(model, X_val, y_val, loss_fn)
        history["val"].append(val_loss)
        if val_loss < history["best_val"]:
            history["best_val"], history["best_epoch"] = val_loss, epoch
            best_state = copy.deepcopy(model.state_dict())     # a snapshot, not a reference
            since_best = 0
        else:
            since_best += 1
            if patience is not None and since_best >= patience:
                break                                           # early stopping
    model.load_state_dict(best_state)                           # hand back the best model, not the last one
    return history


# ---- Lesson 6: the toolkit learns to take a DataLoader ------------------------

def _split_batch(batch, device):
    """A batch is a tuple (input_1, ..., input_k, target). Move everything to ``device``."""
    *inputs, target = batch
    return [x.to(device) for x in inputs], target.to(device)


def train_one_epoch_loader(model: nn.Module, loader, optimizer: torch.optim.Optimizer, loss_fn: nn.Module, device: str | torch.device = "cpu") -> float:
    """One pass over every batch in ``loader``. Returns the mean training loss, weighted by batch size.

    For each batch: split it, zero_grad, forward with ``model(*inputs)``, loss, backward, step.
    """
    raise NotImplementedError("Lesson 6, step 6: loop over loader; accumulate loss.item() * len(target); divide by the number of rows seen")


@torch.no_grad()
def evaluate_loader(model: nn.Module, loader, loss_fn: nn.Module, device: str | torch.device = "cpu") -> float:
    """Mean loss over every batch in ``loader``, weighted by batch size, in eval mode."""
    raise NotImplementedError("Lesson 6, step 6: model.eval(); the same loop without the optimizer")


def fit_loader(model: nn.Module, train_loader, val_loader, epochs: int = 20, lr: float = 1e-3, optimizer_cls=torch.optim.Adam,
               weight_decay: float = 0.0, patience: int | None = None, loss_fn: nn.Module | None = None, device: str | torch.device = "cpu") -> dict:
    """Lesson 5's fit_with_validation, with loaders instead of tensors. Same history, same early stopping, same best-weights rule."""
    raise NotImplementedError("Lesson 6, step 6: copy fit_with_validation and swap in the loader functions; move the model to device first")

In [ ]:
from forkcast import train
importlib.reload(train)
torch.manual_seed(0)
model = TabularNet(4)
history = train.fit_loader(model, train_loader, val_loader, epochs=25, lr=3e-3)
print([round(v, 1) for v in history["val"][:5]])
print("best validation", round(history["best_val"], 2), "at epoch", history["best_epoch"])
plt.plot(history["train"], label="train"); plt.plot(history["val"], label="validation"); plt.yscale("log"); plt.legend();

### Do the zones carry information?

In [ ]:
class ContinuousOnly(nn.Module):           # same MLP, zone ids ignored
    def __init__(self):
        super().__init__()
        self.mlp = MLP(4, (64, 64))
    def forward(self, x_cont, x_cat):
        return self.mlp(x_cont)

torch.manual_seed(0)
without = train.fit_loader(ContinuousOnly(), train_loader, val_loader, epochs=25, lr=3e-3)["best_val"]
print(f"continuous features only:  best validation loss {without:.2f}")
print(f"with zone embeddings:      best validation loss {history['best_val']:.2f}")

### Break it on purpose

Zone ids are indices. Feed them as floats and the embedding refuses.

In [ ]:
try:
    model(cont, cat.float())
except RuntimeError as err:
    print("RuntimeError:", str(err)[:120])

## 6. A real month

Downloads about 50 MB and samples 200,000 trips. Everything above runs unchanged on it; compare the cleaning report and the curves.

In [ ]:
!python tools/get_taxi.py --month 2024-01 --rows 200000
real = trips.load_trips("data/taxi_2024-01.csv")
r_train, r_val, r_test = trips.split_frame(real, seed=0)
r_norm = Normalizer().fit(torch.tensor(r_train[trips.CONTINUOUS].values, dtype=torch.float32))
r_tl, r_vl = trips.make_loaders(r_train, r_val, r_norm, batch_size=512)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
real_model = TabularNet(4)
h = train.fit_loader(real_model, r_tl, r_vl, epochs=10, lr=3e-3, device=device)
print("best validation", round(h["best_val"], 2), "at epoch", h["best_epoch"])
emb = real_model.pu.weight.detach().cpu()
print("zones nearest to JFK (132):", torch.cdist(emb[132:133], emb).squeeze(0).argsort()[:6].tolist())

## 7. Definition of done

Expected: `7 passed`, then `48 passed` with no failures for everything so far (skipped tests belong to lessons you have not started).

In [ ]:
!pytest tests/test_06.py -q

In [ ]:
!pytest -m fast -q

## 8. Commit and push

In [ ]:
from getpass import getpass
TOKEN = getpass("GitHub token with access to your repo: ")

!git config user.name "{GITHUB_USER}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
!git add -A
!git commit -m "Lesson 6: datasets, loaders, embeddings"
!git push https://{TOKEN}@github.com/{GITHUB_USER}/{REPO}.git HEAD:main